TextLoader
PyPdfLoader
WebbasedLoader
arxivLoader
WikipediaLoader
UnstructuredHTMLLoader

Rule of thumb: use JSONLoader when you want jq to pick out specific records as separate Documents (headed toward RecursiveCharacterTextSplitter or straight to embedding). Use RecursiveJsonSplitter when you want to preserve one JSON's nested structure while breaking it into size-bounded pieces — feed it raw dicts via json.load(), not Document.page_content strings from a loader.

# TextLoader

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("speech.txt")
textDoc = loader.load()
print(type(textDoc))
print(len(textDoc))
print(type(textDoc[0]))
print(textDoc[0].metadata)
# print(textDoc[0].page_content)
print(textDoc[0].id)
print(textDoc[0].to_json)



In [ ]:
import sys, os
print(sys.executable)
# print(os.path.exists(sys.executable))

In [ ]:
import sys
!"{sys.executable}" -m pip install pypdf

In [ ]:
import pypdf
print(pypdf.__file__)

# PDF loader

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdfLoader = PyPDFLoader(file_path="attention.pdf")
pdf = pdfLoader.load()
print(type(pdf))
print(pdf[0].page_content)


# arxivloader

In [ ]:
from langchain_community.document_loaders import ArxivLoader

arxivLoader = ArxivLoader(query="1706.03762",doc_content_chars_max="100")
# doc = arxivLoader.load()


In [ ]:
from importlib.metadata import version
print(version("arxiv"))
print(version("langchain_community"))

In [ ]:
import sys
print(sys.executable)

In [ ]:
import requests
import feedparser
from langchain_core.documents import Document

def fetch_arxiv(arxiv_id, max_chars=100):
    url = f"https://export.arxiv.org/api/query?id_list={arxiv_id}"
    resp = requests.get(url)
    resp.raise_for_status()
    feed = feedparser.parse(resp.text)
    
    docs = []
    for entry in feed.entries:
        content = entry.summary[:max_chars] if max_chars else entry.summary
        docs.append(Document(
            page_content=content,
            metadata={
                "Title": entry.title,
                "Authors": ", ".join(a.name for a in entry.authors),
                "Published": entry.published,
            }
        ))
    return docs

doc = fetch_arxiv("1706.03762", max_chars=100)
print(doc)

# DirectoryLoader

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

directory = DirectoryLoader(path="/Users/sanvijanvi/Documents/SJ_HomeWork", glob="**/*.pdf",loader_cls=PyPDFLoader)
docs = directory.load()

for doc in docs:
    print(doc.metadata.get("title"))
    print(doc.metadata.get("source"))


# PyDirectory laoder

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

pdfDirectory = PyPDFDirectoryLoader(path= "/Users/sanvijanvi/Documents/SJ_HomeWork")
pdfDocs = pdfDirectory.load()

for doc in pdfDocs:
    print(doc.metadata.get("source"))

# Interview questions 

Problem 1 <br>

Load a PDF and print:<br>

Page number<br>
Page content length<br>
First 100 characters<br>

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

try :
    pdfLoader = PyPDFLoader(file_path= "/Users/sanvijanvi/Documents/SJ_HomeWork/math-bingo.pdf",mode="page")
    pdfDocs = pdfLoader.load()

    for doc in pdfDocs:
        # print(doc.metadata)
        print(f"Page number : {doc.metadata.get('page')}")
        print(f"Page content length : {len(doc.page_content)}")
        print(f"First 100 characters : {doc.page_content[:100]}")
except Exception as e:
    print(f"failed laoding: /Users/sanvijanvi/Documents/SJ_HomeWork/math-bingo.pdf :{e}")


Problem 2 <br>

Load all .pdf files from:<br>

documents/<br>

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

#document floder has lot of docs, so loading from /Users/sanvijanvi/Documents/SJ_HomeWork/

pdfDirectory = PyPDFDirectoryLoader(path="/Users/sanvijanvi/Documents/SJ_HomeWork/",mode="page") # mode = single gives whole document as 1 doc, page gives number of docs = number of pages in the doc
pdfDocs = pdfDirectory.load()

print(type(pdfDocs))


Problem 3 <br>

Create metadata:<br>

{<br>
    "source": "...",<br>
    "page": ...,<br>
    "document_type": "pdf"<br>
}<br>

for every page.<br>

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdfLoader = PyPDFLoader(file_path="/Users/sanvijanvi/Documents/SJ_HomeWork/math-bingo.pdf")
pdfDocs = pdfLoader.load()

for doc in pdfDocs:
    print(doc.metadata)
    print(f"Source: {doc.metadata.get('source')}")
    print(f"Page: {doc.metadata.get('page')}")
    print(f"document_type: {doc.metadata.get('document_type')}")
    #add you own metadata
    doc.metadata["Developer"] = "Vishala Vikram"

#verify new metada info added
print(pdfDocs[0].metadata.get("Developer"))

Build a loader that accepts:<br>

.txt<br>
.pdf<br>
.html<br>

and automatically selects the appropriate loader.<br>

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, TextLoader,UnstructuredHTMLLoader

loaders = {
    ".txt":TextLoader,
    ".pdf":PyPDFLoader,
    ".html":UnstructuredHTMLLoader
}

docs = []
for extension, loader in loaders.items():
    docLoader = DirectoryLoader(path = "/Users/sanvijanvi/Documents/SJ_HomeWork", glob= f"**/*{extension}",loader_cls=loader,silent_errors=True)
    loaded_docs = docLoader.load()
    docs.extend(loaded_docs)
    print(len(loaded_docs),type(loaded_docs))
    if loaded_docs:
        loaded_docs[0].metadata




Problem 5 Advanced <br>
<br>
Create a document ingestion pipeline that:<br>
<br>
Directory<br>
   ↓<br>
Load files<br>
   ↓<br>
Attach metadata<br>
   ↓<br>
Remove empty documents<br>
   ↓<br>
Return Documents<br>

In [ ]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader,UnstructuredHTMLLoader, DirectoryLoader
from langchain_core.documents import Document

loaders = {
    ".txt":TextLoader,
    ".pdf":PyPDFLoader,
    ".html": UnstructuredHTMLLoader
}


def processDocuments(path,loaders):
  docs = []
  for extension, loader in loaders.items():
    docLoader = DirectoryLoader(path=path,glob=f"**/*{extension}",loader_cls=loader,silent_errors=True)
    loaded_docs = docLoader.load()
    print(extension)
    #attach metadata without modifiying original docs
    meta_updated_docs = [Document(page_content=doc.page_content,metadata = {**doc.metadata,"Developer":"Vishala Vikram"}) for doc in loaded_docs]
    #remove empty docs
    updated_docs = [doc for doc in meta_updated_docs if doc.page_content.strip()]
    docs.extend(updated_docs)
    print(len(updated_docs))
  return docs   

docs = processDocuments(path = "/Users/sanvijanvi/Documents/SJ_HomeWork/",loaders=loaders)
print(len(docs))



In [ ]:
str1 = "df"

print(bool(str1.strip()))